## Data Preproccessing

1. Data is Imbalance(6:4) so we need to balance it.
2. Id ,name and ticket are not required so we will drop it .
3. Sex has two values (male,female) mapped to (0,1)
4. Cabin has huge amount of nulls(approx 70%) so dropping it.
5. Embarked and Pclass will be one hot encoded.

In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer 
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

1. dropping columns

In [3]:
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']

def drop_columns(df, columns):
    return df.drop(columns=columns)

dropper = FunctionTransformer(drop_columns, kw_args={'columns': drop_cols})

2. cerating new column familysize
- `family = sibsp + parch + 1`

In [4]:
def add_family_size(X):
    X = X.copy()
    X["FamilySize"] = X["SibSp"] + X["Parch"] + 1
    return X

# transformer to add family size
family_size_adder = FunctionTransformer(add_family_size)

3. Standerizing the numeric columns anf filling null with median as only age is null
- using mean will not be significant as we want to classify later

4. One hot encoding for categorical features as all hav very few classes

In [5]:
numeric_features = ["Age", "SibSp", "Parch", "Fare", "FamilySize"]
categorical_features = ["Pclass", "Sex", "Embarked"]

# Numeric transformer: fill missing + scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical transformer: fill missing + one-hot encode
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [6]:
# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

#### Final pipeline: drop -> preprocess

In [7]:
full_pipeline = Pipeline(steps=[
    ("drop", dropper),
    ("preprocess", preprocessor)
])

### Using smote to balance the data

In [14]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Final pipeline with preprocessing + SMOTE
smote_pipeline = ImbPipeline(steps=[
    ("preprocessor", full_pipeline),    # your full preprocessing pipeline
    ("smote", SMOTE(random_state=42)) # resampling step
])


In [12]:
# save
joblib.dump(smote_pipeline, "preprocess_smote_pipeline.pkl")

['preprocess_smote_pipeline.pkl']